In [ ]:
import astropy.table as at
import jax
import matplotlib.pyplot as plt
import numpy as np
import quaxed.numpy as jnp
import unxt as u
from astropy.constants import G as _G

import harv

G = u.Q.from_(_G)

jax.config.update("jax_enable_x64", True)

# Multi-survey offsets

In [ ]:
rng = np.random.default_rng(1234)
times1 = rng.uniform(0, 100, 16)
times2 = rng.uniform(50, 150, 12)
times = np.concatenate([times1, times2])

mask = np.zeros(len(times), dtype=bool)
mask[len(times1) :] = True

truth = {
    "P": 24.5234,
    "e": 0.1833,
    "t_peri": 4.139,
    "arg_peri": 51.394,
    "rv_semiamp": 1.3523,
    "v_sys": 40.0,
}
rv = harv.kepler.rv_at_times(
    u.Q(times, "day"),
    u.Q(truth["P"], "day"),
    truth["e"],
    t_peri=u.Q(truth["t_peri"], "day"),
    arg_peri=u.Q(truth["arg_peri"], "deg"),
    rv_semiamp=u.Q(truth["rv_semiamp"], "km/s"),
    v_sys=u.Q(truth["v_sys"], "km/s"),
)
rv = rv.at[mask].set(rv[mask] + u.Q(0.4, "km/s"))

rv_err = u.Q(rng.uniform(0.08, 0.16, len(times)), "km/s")
rv = rv + u.Q(rng.normal(0, rv_err.value), "km/s")

In [ ]:
plt.errorbar(times1, rv[~mask], yerr=rv_err[~mask], fmt="o")
plt.errorbar(times2, rv[mask], yerr=rv_err[mask], fmt="o")

In [ ]:
tbl = at.Table({"time": times, "rv": rv, "rv_err": rv_err})
survey = np.full(len(tbl), "survey1", dtype="U7")
survey[mask] = "survey2"
tbl["survey"] = survey
for k, v in truth.items():
    tbl.meta[k] = v
tbl.write("simulated-multi-survey-data.fits", overwrite=True)

# Triple

In [ ]:
def P_e_M_to_K(P, e, M2, M1):
    """Convert period, eccentricity, and mass to RV semi-amplitude."""
    K = (
        (2 * jnp.pi * G / P) ** (1 / 3)
        * (M2 / (M1 + M2) ** (2 / 3))
        / jnp.sqrt(1 - e**2)
    )
    return K.to("km/s")

In [ ]:
K1 = P_e_M_to_K(u.Q(24.5234, "day"), 0.1833, u.Q(0.2, "Msun"), u.Q(1.0, "Msun"))
K2 = P_e_M_to_K(u.Q(1005.33, "day"), 0.53, u.Q(0.5, "Msun"), u.Q(1.0, "Msun"))
K1, K2

In [ ]:
rng = np.random.default_rng(1234)
times = rng.uniform(0, 300, 21)
# times = rng.uniform(0, 300, 2048)

truth1 = {
    "P": 24.5234,
    "e": 0.1833,
    "t_peri": 4.139,
    "arg_peri": 51.394,
    "rv_semiamp": 13.203,
    "v_sys": 0.0,
}
truth2 = {
    "P": 1005.33,
    "e": 0.53,
    "t_peri": -100.139,
    "arg_peri": 1.394,
    "rv_semiamp": 9.56,
    "v_sys": 0.0,
}

rvs = [
    harv.kepler.rv_at_times(
        u.Q(times, "day"),
        u.Q(truth["P"], "day"),
        truth["e"],
        t_peri=u.Q(truth["t_peri"], "day"),
        arg_peri=u.Q(truth["arg_peri"], "deg"),
        rv_semiamp=u.Q(truth["rv_semiamp"], "km/s"),
        v_sys=u.Q(truth["v_sys"], "km/s"),
    )
    for truth in (truth1, truth2)
]
rv = rvs[0] + rvs[1]

rv_err = u.Q(rng.uniform(0.1, 0.5, len(times)), "km/s")
rv = rv + u.Q(rng.normal(0, rv_err.value), "km/s")

In [ ]:
plt.errorbar(times, rv, rv_err, fmt="o")

In [ ]:
tbl = at.Table({"time": times, "rv": rv, "rv_err": rv_err})
for k, v in truth1.items():
    tbl.meta[f"{k}_1"] = v
for k, v in truth2.items():
    tbl.meta[f"{k}_2"] = v
tbl.write("simulated-long-trend.fits", overwrite=True)